In [12]:
import json
import csv
import requests
import time
import os

# Generate Color Texture for each Coral species
base on the color description of the color species we generate a texture to later us on our point clouds

In [11]:

# Load your exported workflow
with open("comfyUI/color_texture_generation.json", "r") as f:
    workflow_template = json.load(f)

# Open it and look for the KSampler and CLIP Text Encode nodes
POSITIVE_PROMPT_NODE_ID = "2"  
SAVE_IMAGE_NODE_ID = "3"         

COMFYUI_URL = "http://127.0.0.1:8188"
PREPEND = "Full screen texture of "
NEGATIVE_PROMPT = "illustration, animals, humans, objects"

def queue_prompt(workflow):
    payload = {"prompt": workflow}
    response = requests.post(f"{COMFYUI_URL}/prompt", json=payload)
    return response.json()

with open("scrapped_data/coral_species.csv", "r") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        # Clone workflow so we don't mutate the template
        wf = json.loads(json.dumps(workflow_template))

        # Set positive prompt
        positive = row["color"]
        if positive.startswith("Colour:"):
            positive = positive[len("Colour:"):].strip()

        positive = PREPEND + positive
        wf[POSITIVE_PROMPT_NODE_ID]["inputs"]["text"] = positive

        # Set unique save filename
        wf[SAVE_IMAGE_NODE_ID]["inputs"]["filename_prefix"] = f"images/colorTextures/{row['species']}"

        # Queue it
        result = queue_prompt(wf)
        print(f"[{i+1}] Queued: {positive[:60]}... → {result}")

        # Small delay to avoid flooding the queue
        time.sleep(1)

print("All prompts queued!")

[1] Queued: Full screen texture of Usually pale grey, brown or rust colo... → {'prompt_id': '632ff34b-4e0d-42ab-a373-a00ee7893f62', 'number': 39, 'node_errors': {}}
[2] Queued: Full screen texture of Uniform or mottled brown, yellow or g... → {'prompt_id': 'eb8adc0e-291a-4f0d-acd3-fc3a60c0f64d', 'number': 40, 'node_errors': {}}
[3] Queued: Full screen texture of Uniform or mottled dull brown, grey o... → {'prompt_id': 'bb9804e4-02f7-4413-abce-ddea8abbd9c6', 'number': 41, 'node_errors': {}}
[4] Queued: Full screen texture of Mottled browns.... → {'prompt_id': '7906bda0-b1e6-4746-a74a-995ce1400860', 'number': 42, 'node_errors': {}}
[5] Queued: Full screen texture of Mottled browns and greens, commonly w... → {'prompt_id': '373ad44e-a31b-43a7-a858-77849671405e', 'number': 43, 'node_errors': {}}
[6] Queued: Full screen texture of Red, cream and brown, with walls and ... → {'prompt_id': 'ed9803fd-298d-4e0a-9d99-f51092a01ff2', 'number': 44, 'node_errors': {}}
[7] Queued: Full screen texture 

# Generate depth image for each hand image
this will later be used to create the point clouds

In [ ]:
with open("comfyUI/depth_image_generator.json", "r") as f:
    workflow_template = json.load(f)

COMFYUI_URL = "http://127.0.0.1:8188"
IMAGES_FOLDER = "/hand_images"  # change this
VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}

def queue_prompt(workflow):
    payload = {"prompt": workflow}
    response = requests.post(f"{COMFYUI_URL}/prompt", json=payload)
    return response.json()

image_files = [
    f for f in os.listdir(IMAGES_FOLDER)
    if os.path.splitext(f)[1].lower() in VALID_EXTENSIONS
]

print(f"Found {len(image_files)} images")

for i, image_filename in enumerate(image_files):
    if i>3:
        break
    wf = json.loads(json.dumps(workflow_template))

    wf["3"]["inputs"]["image"] = image_filename

    name_without_ext = os.path.splitext(image_filename)[0]
    wf["2"]["inputs"]["filename_prefix"] = f"hand_images_depth/{name_without_ext}"

    result = queue_prompt(wf)
    print(f"[{i+1}/{len(image_files)}] Queued: {image_filename} → {result}")

    time.sleep(1)

print("All depth maps queued!")